# stml — EDA: OHLCV prices & primary signals

Goal: build intuition for the two datasets and — most importantly — their **joint** behaviour, so that downstream modelling decisions (label balance, horizon, regime handling, look-ahead risk) are made with eyes open.

**Sections**
1. Inventory & coverage
2. OHLCV — per-instrument structure
3. Cross-sectional structure of prices
4. Signals — structure
5. Joint: signals × forward returns (the payoff section)
6. Takeaways

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

sns.set_theme(context='notebook', style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)

DATA = Path('..') / 'data'
INSTRUMENTS = ['cl1s','es1s','fesx1s','gc1s','hg1s','ho1s','ng1s','nq1s','pl1s','rb1s','si1s']

# Friendly labels (helps reading charts)
TICKER_NAMES = {
    'cl1s': 'Crude Oil (CL)',
    'es1s': 'S&P 500 e-mini (ES)',
    'fesx1s': 'Euro Stoxx 50 (FESX)',
    'gc1s': 'Gold (GC)',
    'hg1s': 'Copper (HG)',
    'ho1s': 'Heating Oil (HO)',
    'ng1s': 'Natural Gas (NG)',
    'nq1s': 'Nasdaq 100 (NQ)',
    'pl1s': 'Platinum (PL)',
    'rb1s': 'RBOB Gasoline (RB)',
    'si1s': 'Silver (SI)',
}

In [ ]:
ohlcv = pd.read_csv(DATA / 'ohlcv_data.csv', parse_dates=['date'])
signals = pd.read_csv(DATA / 'primary_signals.csv', parse_dates=['date'])

# Sort canonically
ohlcv = ohlcv.sort_values(['instrument', 'date']).reset_index(drop=True)
signals = signals.sort_values('date').reset_index(drop=True)

# Wide pivots — useful for cross-sectional work
close_w  = ohlcv.pivot(index='date', columns='instrument', values='close')[INSTRUMENTS]
volume_w = ohlcv.pivot(index='date', columns='instrument', values='volume')[INSTRUMENTS]
oi_w     = ohlcv.pivot(index='date', columns='instrument', values='open_interest')[INSTRUMENTS]

# Log returns per instrument (wide)
ret_w = np.log(close_w).diff()

# Signal frame indexed by date for easy joins
sig_w = signals.set_index('date')[INSTRUMENTS]

print(f'OHLCV: {ohlcv.shape[0]:,} rows × {ohlcv.shape[1]} cols   |   '
      f'{ohlcv.date.min().date()} → {ohlcv.date.max().date()}')
print(f'Signals: {signals.shape[0]:,} rows × {signals.shape[1]} cols   |   '
      f'{signals.date.min().date()} → {signals.date.max().date()}')

## 1. Inventory & coverage

Before anything else: what is *actually* in each file, how long is the history per instrument, and how does the signal window sit inside the price window?

In [ ]:
inv = (ohlcv
       .groupby('instrument')
       .agg(start=('date', 'min'),
            end=('date', 'max'),
            rows=('date', 'size'),
            mean_close=('close', 'mean'),
            median_volume=('volume', 'median'),
            mean_oi=('open_interest', 'mean'))
       .reindex(INSTRUMENTS))
inv['years'] = ((inv['end'] - inv['start']).dt.days / 365.25).round(1)
inv['name'] = inv.index.map(TICKER_NAMES)
inv = inv[['name', 'start', 'end', 'years', 'rows', 'mean_close', 'median_volume', 'mean_oi']]
inv

In [ ]:
# Data-availability heatmap (one row per instrument, time on x)
avail = close_w.notna().astype(int).T  # instrument × date
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.imshow(avail.values, aspect='auto', cmap='Greys', interpolation='nearest')
ax.set_yticks(range(len(avail.index)))
ax.set_yticklabels(avail.index)
# x ticks at year boundaries
yrs = pd.to_datetime(avail.columns).year
yr_changes = np.where(np.diff(yrs) != 0)[0] + 1
step = max(1, len(yr_changes) // 12)
ax.set_xticks(yr_changes[::step])
ax.set_xticklabels([str(yrs[i]) for i in yr_changes[::step]], rotation=45)
# Mark signal window
sig_start_idx = avail.columns.get_indexer([signals.date.min()])[0]
sig_end_idx   = avail.columns.get_indexer([signals.date.max()])[0]
ax.axvline(sig_start_idx, color='C3', lw=1.2, label='signal window')
ax.axvline(sig_end_idx,   color='C3', lw=1.2)
ax.set_title('OHLCV data availability per instrument  (red lines = signal window)')
ax.legend(loc='lower left')
plt.tight_layout(); plt.show()

In [ ]:
# Signal-side coverage check: for each signal date, how many of the 11 instruments have OHLCV?
sig_dates = signals['date'].unique()
cov = (close_w.loc[sig_dates]
       .notna()
       .sum(axis=1)
       .value_counts()
       .sort_index(ascending=False))
print('Signal-date coverage (# instruments with OHLCV that day → # signal days):')
print(cov)
print('\nAll signal dates ∈ OHLCV dates?', set(sig_dates).issubset(close_w.index))

**Read:** 11 instruments. 7 commodities (`cl, ho, rb, ng, gc, si, pl, hg`) start in 1990 with ~32 years of daily bars; equity-index futures (`es, nq, fesx`) start later (mid-90s onward). The signal file only covers **2020–2022** — a tiny fraction of price history, and squarely inside the COVID-driven vol regime.

## 2. OHLCV — per-instrument structure

In [ ]:
# Price small-multiples (log scale)
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=True)
for ax, inst in zip(axes.flat, INSTRUMENTS):
    s = close_w[inst].dropna()
    ax.plot(s.index, s.values, lw=0.8)
    ax.set_yscale('log')
    ax.set_title(TICKER_NAMES[inst], fontsize=10)
    ax.grid(True, alpha=0.3)
for ax in axes.flat[len(INSTRUMENTS):]:
    ax.set_visible(False)
fig.suptitle('Close prices (log scale)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Volume & open-interest small-multiples — useful for spotting electronification / liquidity shifts
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=True)
for ax, inst in zip(axes.flat, INSTRUMENTS):
    v = volume_w[inst].dropna()
    oi = oi_w[inst].dropna()
    ax.plot(v.index, v.values, lw=0.6, label='volume', color='C0')
    ax2 = ax.twinx()
    ax2.plot(oi.index, oi.values, lw=0.6, label='open interest', color='C3', alpha=0.7)
    ax.set_title(TICKER_NAMES[inst], fontsize=10)
    ax.tick_params(axis='y', labelsize=7)
    ax2.tick_params(axis='y', labelsize=7, colors='C3')
for ax in axes.flat[len(INSTRUMENTS):]:
    ax.set_visible(False)
fig.suptitle('Volume (blue) and open interest (red)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Returns summary: mean, std (annualised), skew, kurtosis
ANN = np.sqrt(252)
ret_summary = pd.DataFrame({
    'mean_daily_bp': ret_w.mean() * 1e4,
    'ann_vol_%':     ret_w.std() * ANN * 100,
    'skew':          ret_w.skew(),
    'kurtosis':      ret_w.kurt(),
    'min_day_%':     ret_w.min() * 100,
    'max_day_%':     ret_w.max() * 100,
    'n_obs':         ret_w.count(),
}).round(2)
ret_summary

In [ ]:
# Returns distribution — overlay normal for comparison
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, inst in zip(axes.flat, INSTRUMENTS):
    r = ret_w[inst].dropna()
    ax.hist(r, bins=80, density=True, alpha=0.7, color='C0')
    xs = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    ax.plot(xs, (1 / (r.std() * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((xs - r.mean())/r.std())**2),
            color='C3', lw=1.2, label='N(μ,σ²)')
    ax.set_title(f'{inst}  kurt={r.kurt():.1f}', fontsize=10)
    ax.set_yscale('log')
    ax.legend(fontsize=7)
for ax in axes.flat[len(INSTRUMENTS):]:
    ax.set_visible(False)
fig.suptitle('Log-return distributions (log y, Gaussian overlay) — fat tails everywhere', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 60-day annualised vol, all on one chart
rolling_vol = ret_w.rolling(60).std() * ANN * 100
fig, ax = plt.subplots(figsize=(13, 5))
rolling_vol.plot(ax=ax, lw=0.8, alpha=0.8)
ax.set_title('Rolling 60-day annualised volatility (%)')
ax.set_ylabel('% vol'); ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

We see a lot of gaps because the rolling method produces NA values if one of the values from the row is missing.

Since these contracts have different trading date alignments, naively applying the rolling method to dataframe doesn't help much.

A remedy would be to apply rolling vol individually and then aggregate it.

In [ ]:
rolling_vol = (
    ohlcv.sort_values(["instrument", "date"])
    .assign(log_close=lambda x: np.log(x["close"]))
    .assign(ret=lambda x: x.groupby("instrument")["log_close"].diff())
    .set_index("date")
    .groupby("instrument")["ret"]
    .rolling(60)
    .std()
    .mul(np.sqrt(252) * 100)
    .rename("rolling_vol")
    .reset_index()
    .pivot(index="date", columns="instrument", values="rolling_vol")
    [INSTRUMENTS]
)

# Rolling 60-day annualised vol, all on one chart
fig, ax = plt.subplots(figsize=(13, 5))
rolling_vol.plot(ax=ax, lw=0.8, alpha=0.8)
ax.set_title('Rolling 60-day annualised volatility (%)')
ax.set_ylabel('% vol'); ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Max drawdown per instrument (on cumulative log return)
cum = ret_w.cumsum()
dd = cum - cum.cummax()  # in log-return units
dd_pct = (np.exp(dd) - 1) * 100  # convert to %
mdd = pd.DataFrame({
    'max_drawdown_%': dd_pct.min().round(1),
    'mdd_date':       dd_pct.idxmin(),
}).sort_values('max_drawdown_%')
mdd

In [ ]:
# ADF on log returns per instrument — expect stationarity
adf_rows = []
for inst in INSTRUMENTS:
    r = ret_w[inst].dropna().values
    stat, pval, *_ = adfuller(r, autolag='AIC')
    adf_rows.append({'instrument': inst, 'adf_stat': stat, 'p_value': pval})
adf_df = pd.DataFrame(adf_rows).set_index('instrument').round(4)
adf_df['stationary_at_5%'] = adf_df['p_value'] < 0.05
adf_df

**Read:** all return series reject the unit-root null (stationary). Kurtosis is well above 3 across the board — fat tails are the rule, not the exception. Vol regimes are obvious: 2008 GFC, 2014–15 oil collapse, 2020 COVID, 2022 energy spike.

## 3. Cross-sectional structure of prices

In [ ]:
# Correlation of daily log returns, full sample (pairwise complete)
corr = ret_w.corr()
g = sns.clustermap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(8, 8), cbar_pos=(0.02, 0.85, 0.03, 0.12))
g.fig.suptitle('Daily log-return correlation (clustered) — energy, metals, equity blocks', y=1.02)
plt.show()

In [ ]:
# Rolling 252-day correlation for a few sentinel pairs
pairs = [('gc1s', 'si1s'), ('cl1s', 'ho1s'), ('es1s', 'nq1s'), ('gc1s', 'es1s')]
fig, ax = plt.subplots(figsize=(13, 4.5))
for a, b in pairs:
    rc = ret_w[a].rolling(252).corr(ret_w[b])
    ax.plot(rc.index, rc.values, lw=1.0, label=f'{a} vs {b}')
ax.axhline(0, color='k', lw=0.5)
ax.set_ylim(-0.4, 1.0)
ax.set_title('Rolling 252-day return correlation (within and across sectors)')
ax.legend(); plt.tight_layout(); plt.show()

Because `ret_w` was created by a wide pivot using a union of dates, this creates lots of NaNs. Thus this generates an empty rolling corr plot. 

We will fix this by computing rolling correlation on *intersection* of data.

In [ ]:
test_pair = ('gc1s', 'si1s')
test = (ohlcv["instrument"] == 'gc1s')
ohlcv[test]


In [ ]:
# write a function that computes intersection of date indices given a pair
def get_intersected_dates(df, pair):
    """
    Parameters:
        df : pd.DataFrame
        pair : tuple (length 2)
    """
    for sec in pair:
        if sec not in df["instrument"].unique():
            raise ValueError(f"{sec} not in instruments")
    secA = pair[0]
    secB = pair[1]
    datesA = df[df["instrument"]==secA]["date"]
    datesB = df[df["instrument"]==secB]["date"]

    final_dates = set(datesA.values).intersection(set(datesB.values))
    final_dates = np.array(sorted(final_dates))
    
    return final_dates

# test
pair_dates = get_intersected_dates(ohlcv, ('gc1s', 'si1s'))

In [ ]:
flag_rows_with_na = ret_w.loc[pair_dates,['gc1s', 'si1s']].isna().any(axis=1)
ret_w.loc[pair_dates,['gc1s', 'si1s']].loc[flag_rows_with_na]

In [ ]:
ret_w.loc["2021-11-15":"2021-12-01", ["gc1s", "si1s"]]

In [ ]:
ret_w

**Read:** the clustermap surfaces three blocks — energy (`cl, ho, rb, ng`), precious/industrial metals (`gc, si, pl, hg`), equity indices (`es, nq, fesx`). The rolling chart shows that intra-block correlations (e.g. `cl/ho`, `es/nq`) are persistently high (~0.8+), while cross-block correlations (`gc/es`) flip sign across regimes — a portfolio built on full-sample correlation will mis-state risk in any individual year.

## 4. Signals — structure

In [ ]:
# Class balance per instrument
balance = (sig_w.apply(lambda s: s.value_counts())
                .reindex([-1, 0, 1])
                .fillna(0).astype(int)
                .T)
balance.columns = ['short (-1)', 'flat (0)', 'long (+1)']
balance['n'] = balance.sum(axis=1)
balance['long_share'] = (balance['long (+1)']  / balance['n']).round(2)
balance['flat_share'] = (balance['flat (0)']   / balance['n']).round(2)
balance['short_share']= (balance['short (-1)'] / balance['n']).round(2)
balance

In [ ]:
# Stacked bar of class shares
fig, ax = plt.subplots(figsize=(11, 4))
shares = balance[['short_share', 'flat_share', 'long_share']]
shares.plot(kind='bar', stacked=True, ax=ax,
            color=['#d62728', '#bdbdbd', '#2ca02c'])
ax.set_ylabel('share of days'); ax.set_ylim(0, 1)
ax.set_title('Signal class balance per instrument (2020–2022)')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout(); plt.show()

In [ ]:
# Signals as a heatmap through time
fig, ax = plt.subplots(figsize=(13, 4.2))
im = ax.imshow(sig_w.T.values, aspect='auto', cmap='RdYlGn',
               vmin=-1, vmax=1, interpolation='nearest')
ax.set_yticks(range(len(INSTRUMENTS))); ax.set_yticklabels(INSTRUMENTS)
# x ticks at month boundaries
months = pd.to_datetime(sig_w.index).to_period('M')
first_of_month = np.where(months != np.roll(months, 1))[0]
step = max(1, len(first_of_month) // 10)
ax.set_xticks(first_of_month[::step])
ax.set_xticklabels([str(sig_w.index[i].date()) for i in first_of_month[::step]], rotation=45)
ax.set_title('Signal heatmap  (red = short, grey = flat, green = long)')
plt.colorbar(im, ax=ax, ticks=[-1, 0, 1], shrink=0.7)
plt.tight_layout(); plt.show()

In [ ]:
# Persistence: mean run length per label, daily flip rate, lag-1 autocorrelation
def run_lengths(s):
    grp = (s != s.shift()).cumsum()
    return s.groupby(grp).size()

rows = []
for inst in INSTRUMENTS:
    s = sig_w[inst]
    rl = pd.DataFrame({'sig': s.values, 'run': run_lengths(s).reindex(((s != s.shift()).cumsum()).values).values})
    by_label = rl.groupby('sig')['run'].mean()
    rows.append({
        'instrument': inst,
        'mean_run_short': by_label.get(-1, np.nan),
        'mean_run_flat':  by_label.get(0,  np.nan),
        'mean_run_long':  by_label.get(1,  np.nan),
        'flip_rate':      (s != s.shift()).iloc[1:].mean(),
        'lag1_autocorr':  s.autocorr(lag=1),
    })
persistence = pd.DataFrame(rows).set_index('instrument').round(2)
persistence

In [ ]:
# Cross-instrument signal correlation (treat ±1/0 as numeric)
sig_corr = sig_w.corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(sig_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Cross-instrument signal correlation')
plt.tight_layout(); plt.show()

In [ ]:
# Daily net position: # longs - # shorts across the 11 instruments
net = sig_w.sum(axis=1)
n_long  = (sig_w ==  1).sum(axis=1)
n_short = (sig_w == -1).sum(axis=1)
n_flat  = (sig_w ==  0).sum(axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax1.fill_between(n_long.index,  0,  n_long,  color='C2', alpha=0.7, label='# long')
ax1.fill_between(n_short.index, 0, -n_short, color='C3', alpha=0.7, label='# short')
ax1.plot(n_flat.index, n_flat, color='gray', lw=0.8, label='# flat')
ax1.axhline(0, color='k', lw=0.5); ax1.legend(loc='upper left')
ax1.set_title('Daily count of long / short / flat positions across 11 instruments')

ax2.plot(net.index, net.rolling(20).mean(), color='k', lw=1.0)
ax2.axhline(0, color='gray', lw=0.5)
ax2.set_title('Net position (longs − shorts), 20-day rolling mean — risk-on / risk-off gauge')
plt.tight_layout(); plt.show()

**Read:** signals are highly imbalanced and *very* instrument-specific. `ho1s` is flat ~90% of the time; `ng1s` never goes long; `es1s` is long ~70% of days. Lag-1 autocorrelation is high (often 0.7+) — the signal is a slow-moving regime call, not a noisy day-to-day flip — so naïve random shuffling for cross-validation will leak. Net position drifts visibly (risk-on through 2020 recovery, risk-off in 2022).

## 5. Joint: signals × forward returns

Does the signal predict next-day (and 5d, 20d) returns? If it doesn't, no amount of downstream modelling will help.

In [ ]:
# Align signals with forward log returns. Convention: signal_t * r_{t+h}  (no look-ahead on the signal).
def fwd_logret(h):
    # log return from close_t to close_{t+h} = ret_{t+1} + ... + ret_{t+h}
    return ret_w.shift(-h).rolling(h).sum().shift(- (h - h))  # simplified below
# Cleaner equivalents:
fwd1  = ret_w.shift(-1)
fwd5  = ret_w.rolling(5).sum().shift(-5)
fwd20 = ret_w.rolling(20).sum().shift(-20)

# Restrict to signal dates only
fwd1_s  = fwd1.loc[sig_w.index]
fwd5_s  = fwd5.loc[sig_w.index]
fwd20_s = fwd20.loc[sig_w.index]
print('Aligned shapes:', fwd1_s.shape, fwd5_s.shape, fwd20_s.shape)

In [ ]:
# Per-instrument: mean forward return by signal label, plus naive Sharpe of signal_t * fwd1_t
def summarise(fwd, h_label):
    rows = []
    for inst in INSTRUMENTS:
        sig = sig_w[inst]
        r = fwd[inst]
        df = pd.concat([sig.rename('sig'), r.rename('r')], axis=1, sort=False).dropna()
        by = df.groupby('sig')['r'].mean()
        pnl = (df['sig'] * df['r']).dropna()
        sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 / 1) if pnl.std() > 0 else np.nan
        # hit rate counted on ±1 signals only
        active = df[df['sig'] != 0]
        hit = (np.sign(active['r']) == np.sign(active['sig'])).mean() if len(active) else np.nan
        rows.append({
            'instrument': inst,
            f'mean_r_short_{h_label}_bp': by.get(-1, np.nan) * 1e4,
            f'mean_r_flat_{h_label}_bp':  by.get(0,  np.nan) * 1e4,
            f'mean_r_long_{h_label}_bp':  by.get(1,  np.nan) * 1e4,
            f'hit_rate_{h_label}':        hit,
            f'pnl_sharpe_{h_label}':      sharpe,
        })
    return pd.DataFrame(rows).set_index('instrument').round(3)

joint1 = summarise(fwd1, '1d')
joint5 = summarise(fwd5, '5d')
joint20 = summarise(fwd20, '20d')
joint = pd.concat([joint1, joint5, joint20], axis=1)
joint

In [ ]:
# Boxplot of 1-day forward returns conditioned on signal label, per instrument
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharey=False)
PALETTE = {'-1': '#d62728', '0': '#bdbdbd', '1': '#2ca02c'}
for ax, inst in zip(axes.flat, INSTRUMENTS):
    df = pd.concat([sig_w[inst].rename('sig'),
                    (fwd1_s[inst] * 100).rename('r%')], axis=1).dropna()
    if df['sig'].nunique() < 2:
        ax.set_visible(False); continue
    df['sig'] = df['sig'].astype(int).astype(str)
    order = [k for k in ['-1', '0', '1'] if k in df['sig'].unique()]
    sns.boxplot(data=df, x='sig', y='r%', ax=ax,
                order=order, hue='sig', hue_order=order,
                palette={k: PALETTE[k] for k in order},
                legend=False, showfliers=False)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(inst, fontsize=10); ax.set_xlabel(''); ax.set_ylabel('next-day %')
for ax in axes.flat[len(INSTRUMENTS):]:
    ax.set_visible(False)
fig.suptitle('Next-day return conditioned on signal label  (want: long > flat > short)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Equity curves: position = signal, daily pnl = sig_t * r_{t+1}; equal-weight portfolio = mean across instruments
pnl = sig_w * fwd1_s  # per-instrument daily pnl in log-return units
pnl = pnl.dropna(how='all')
cum_pnl = pnl.cumsum()
portfolio = pnl.mean(axis=1).cumsum()  # equal-weight

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
cum_pnl.plot(ax=ax1, lw=0.9, alpha=0.85)
ax1.axhline(0, color='k', lw=0.5)
ax1.set_title('Cumulative pnl per instrument  (signal × next-day return, log-return units)')
ax1.legend(ncol=2, fontsize=8)

portfolio.plot(ax=ax2, lw=1.6, color='k', label='equal-weight portfolio')
# overlay buy-and-hold of each underlying over the same window
for inst in INSTRUMENTS:
    bh = ret_w.loc[sig_w.index, inst].fillna(0).cumsum()
    ax2.plot(bh.index, bh.values, lw=0.6, alpha=0.5, label=f'B&H {inst}')
ax2.axhline(0, color='k', lw=0.5)
ax2.set_title('Strategy vs buy-and-hold of each underlying')
ax2.legend(ncol=2, fontsize=7)
plt.tight_layout(); plt.show()

# Portfolio summary
p_daily = pnl.mean(axis=1)
print(f'\nEqual-weight portfolio: ann. mean = {p_daily.mean()*252*100:.2f}%   '
      f'ann. vol = {p_daily.std()*np.sqrt(252)*100:.2f}%   '
      f'Sharpe = {(p_daily.mean()/p_daily.std())*np.sqrt(252):.2f}')

In [ ]:
# Lead/lag: corr(signal_t, r_{t+k}) for k in [-5..5]  (k>0 = signal leads return = predictive)
lags = range(-5, 6)
lead_lag = pd.DataFrame(index=lags, columns=INSTRUMENTS, dtype=float)
for k in lags:
    shifted = ret_w.shift(-k)  # r_{t+k}
    aligned = shifted.loc[sig_w.index]
    for inst in INSTRUMENTS:
        a = sig_w[inst].astype(float)
        b = aligned[inst]
        ok = a.notna() & b.notna()
        if ok.sum() > 30 and a[ok].std() > 0 and b[ok].std() > 0:
            lead_lag.loc[k, inst] = a[ok].corr(b[ok])

fig, ax = plt.subplots(figsize=(11, 5))
for inst in INSTRUMENTS:
    ax.plot(lead_lag.index, lead_lag[inst], marker='o', lw=1.0, label=inst)
ax.axvline(0, color='k', lw=0.5); ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('lag k:  k>0 ⇒ signal leads return  (predictive)')
ax.set_ylabel('corr(signal_t, r_{t+k})')
ax.set_title('Lead/lag: is the signal predictive (k>0) or reactive (k<0)?')
ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Conditional vol: is the signal more confident in calm or stormy markets?
rows = []
for inst in INSTRUMENTS:
    df = pd.concat([sig_w[inst].rename('sig'),
                    fwd1_s[inst].rename('r')], axis=1).dropna()
    vol_active = df.loc[df['sig'] != 0, 'r'].std() * np.sqrt(252) * 100
    vol_flat   = df.loc[df['sig'] == 0, 'r'].std() * np.sqrt(252) * 100
    rows.append({'instrument': inst,
                 'ann_vol_when_active_%': vol_active,
                 'ann_vol_when_flat_%':   vol_flat,
                 'ratio_active_over_flat': vol_active / vol_flat if vol_flat else np.nan})
cond_vol = pd.DataFrame(rows).set_index('instrument').round(2)
cond_vol

## 6. Takeaways for downstream modelling

- **Two very different windows.** OHLCV gives ~32 years of history; the signal only covers 2.5 years (2020-01 → 2022-06), and that window is dominated by COVID + the 2022 energy spike. Don't claim out-of-sample performance from a window this short.
- **Heavy class imbalance, per-instrument.** `ho1s` is flat 90% of the time; `ng1s` is never long; `es1s` is long 70% of the time. Any uniform classifier head will be misleading — calibrate per instrument and watch for trivial-class collapse.
- **Signals are persistent, not noisy.** Lag-1 autocorrelation 0.6–0.9, mean run lengths often >5 days. **Random k-fold CV will leak**; use blocked / purged time-series splits.
- **The dataset is panel-structured.** Returns within sectors (energy / metals / equity) correlate 0.7–0.9; cross-sector correlations swing sign by regime. Treat instruments as a *panel* (shared features + per-instrument heads) rather than 11 independent univariate problems.
- **Look-ahead pitfall.** Signal is labelled on date `t`; the natural target is `r_{t+1}`. Don't accidentally regress on `r_t` (= contemporaneous close, which the signal could partly reflect).
- **Survivorship / start-dates.** `es, nq, fesx` start later — full-panel models must handle ragged starts or restrict to the common window.
- **Fat tails.** Kurtosis ≫ 3 everywhere. Mean-squared-error losses on raw returns get dragged by tails; consider Huber loss, log-returns (already used here), or volatility-normalised targets.